In [41]:
%matplotlib tk
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots
from scipy.integrate import solve_ivp

plt.style.use(['science','notebook', 'grid'])

In [129]:
# PARARMS TO CHANGE
m3 = 1
v1 =  0.39295
v2 = 0.09758

# Everything else follows from paper
m1 = 1
m2 = 1 
m3 = m3
x1_0 = -1
y1_0 = 0
x2_0 = 1
y2_0 = 0
x3_0 = 0
y3_0 = 0
vx1_0 =  v1
vy1_0 =  v2
vx2_0 = v1
vy2_0 = v2
vx3_0 = -2*v1/m3+0.5
vy3_0 = -2*v2/m3

In [130]:
def dSdt(t, S):
    x1, y1, x2, y2, x3, y3, vx1, vy1, vx2, vy2, vx3, vy3 = S
    r12 = np.sqrt((x2-x1)**2 + (y2-y1)**2)
    r13 = np.sqrt((x3-x1)**2 + (y3-y1)**2)
    r23 = np.sqrt((x2-x3)**2 + (y2-y3)**2)
    return [ vx1,
            vy1,
            vx2,
            vy2,
            vx3,
            vy3,
            m2/r12**3 * (x2-x1) + m3/r13**3 * (x3-x1), #mass 1
            m2/r12**3 * (y2-y1) + m3/r13**3 * (y3-y1),
            m1/r12**3 * (x1-x2) + m3/r23**3 * (x3-x2), #mass 2
            m1/r12**3 * (y1-y2) + m3/r23**3 * (y3-y2),
            m1/r13**3 * (x1-x3) + m2/r23**3 * (x2-x3), #mass 3
            m1/r13**3 * (y1-y3) + m2/r23**3 * (y2-y3)
           ]

In [131]:
t = np.linspace(0, 200, 10000)

sol = solve_ivp(dSdt, (0,200), y0=[x1_0, y1_0, x2_0, y2_0, x3_0, y3_0,
                       vx1_0, vy1_0, vx2_0, vy2_0, vx3_0, vy3_0],
                method = 'DOP853', t_eval=t, rtol=1e-10, atol=1e-13)

In [132]:
t = sol.t
x1 = sol.y[0]
y1 = sol.y[1]
x2 = sol.y[2]
y2 = sol.y[3]
x3 = sol.y[4]
y3 = sol.y[5]

In [133]:
plt.plot(t,x1)
tt = 1/np.sqrt(6.67e-11 * 1.99e30 / (1.5e11)**3 ) # seconds
tt = tt / (60*60 * 24* 365.25) * np.diff(t)[0] # per time step (in years)

In [134]:
from matplotlib import pyplot as plt, animation

def animate(i):
    ln1.set_data([x1[i], x2[i], x3[i]], [y1[i], y2[i], y3[i]])
    text.set_text('Time = {:.1f} Years'.format(i*tt))
fig, ax = plt.subplots(1,1, figsize=(6,6))
ax.grid()
ln1, = plt.plot([], [], 'ro', lw=3, markersize=6)
text = plt.text(0, 19.5, 'asdasd', fontsize=20, backgroundcolor='white', ha='center')
ax.set_ylim(-2, 2)
ax.set_xlim(-2,2)
ani = animation.FuncAnimation(fig, animate, frames=1000, interval=50)
print(np.average(avg_dist(x1, y1, x2, y2, x3, y3)))

19.160982821508505


In [138]:
# PARARMS TO CHANGE
m3 = 1
v1 =  0.39295
v2 = 0.09758

# Everything else follows from paper
m1 = 1
m2 = 1 
m3 = m3
x1_0 = -1
y1_0 = 0
x2_0 = 1
y2_0 = 0
x3_0 = 0
y3_0 = 0
vx1_0 =  v1
vy1_0 =  v2
vx2_0 = v1
vy2_0 = v2
vx3_0 = -2*v1/m3
vy3_0 = -2*v2/m3

In [139]:
def avg_dist(x1, y1, x2, y2, x3, y3):
    r12 = np.sqrt((x2-x1)**2 + (y2-y1)**2)
    r13 = np.sqrt((x3-x1)**2 + (y3-y1)**2)
    r23 = np.sqrt((x2-x3)**2 + (y2-y3)**2)
    return np.average([r12, r13, r23])

In [140]:
t = np.linspace(0, 20, 1000)
vx = np.linspace(-1, 1, 6)
vy = np.linspace(-1, 1, 6)
VX, VY = np.meshgrid(vx, vy, indexing='ij')
Z = np.zeros_like(VX)
for i in range(np.shape(VX)[0]):
    for j in range(np.shape(VY)[1]):
        sol = solve_ivp(dSdt, (0,200), y0=[x1_0, y1_0, x2_0, y2_0, x3_0, y3_0,
                       vx1_0, vy1_0, vx2_0, vy2_0, vx3_0 + VX[i, j], vy3_0 + VY[i, j]],
                method = 'DOP853', t_eval=t, rtol=1e-10, atol=1e-13)
        x1 = sol.y[0]
        y1 = sol.y[1]
        x2 = sol.y[2]
        y2 = sol.y[3]
        x3 = sol.y[4]
        y3 = sol.y[5]
        Z[i,j] = np.average(avg_dist(x1[-100:], y1[-100:], x2[-100:], y2[-100:], x3[-100:], y3[-100:]))


(6, 6)

In [144]:
Z = 1/Z

In [142]:
plt.contourf(vx+vx3_0, vy+vy3_0, Z, levels= 10)
plt.colorbar()

In [145]:
plt.imshow(Z)